# Optimized RAG Implementation - CPU/GPU Compatible

This notebook provides an optimized RAG (Retrieval Augmented Generation) implementation that automatically adapts to available hardware (CPU or GPU).

## Key Optimizations:
- **Automatic device detection** - Uses GPU if available, falls back to CPU
- **Mixed precision inference** - FP16/BF16 on GPU for faster processing
- **Optimized batch processing** - Configurable batch sizes based on device
- **Memory-efficient embedding** - Gradient checkpointing and memory management
- **Async processing** - Non-blocking operations for better throughput
- **Performance benchmarking** - Track and compare CPU vs GPU performance

## 1. Install Dependencies

In [ ]:
# Uncomment to install required packages
# !pip install llama-index llama-index-vector-stores-lancedb llama-index-embeddings-huggingface -q
# !pip install llama-index-llms-huggingface-api llama-index-llms-ollama lancedb datasets -q
# !pip install torch accelerate transformers sentence-transformers -q

## 2. Device Detection and Configuration

In [ ]:
import os
import sys
import time
import gc
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Literal
from functools import wraps

import torch
import numpy as np

warnings.filterwarnings('ignore')

@dataclass
class DeviceConfig:
    """Configuration for device-aware processing"""
    device: str
    device_name: str
    is_cuda: bool
    is_mps: bool  # Apple Silicon
    is_cpu: bool
    dtype: torch.dtype
    batch_size: int
    num_workers: int
    memory_gb: float

def detect_device() -> DeviceConfig:
    """
    Detect available hardware and return optimal configuration.
    Supports CUDA (NVIDIA), MPS (Apple Silicon), and CPU.
    """
    if torch.cuda.is_available():
        device = "cuda"
        device_name = torch.cuda.get_device_name(0)
        memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        
        # Choose dtype based on GPU capability
        if torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16
        else:
            dtype = torch.float16
        
        # Adjust batch size based on GPU memory
        if memory_gb >= 16:
            batch_size = 64
        elif memory_gb >= 8:
            batch_size = 32
        else:
            batch_size = 16
            
        return DeviceConfig(
            device=device,
            device_name=device_name,
            is_cuda=True,
            is_mps=False,
            is_cpu=False,
            dtype=dtype,
            batch_size=batch_size,
            num_workers=4,
            memory_gb=memory_gb
        )
    
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        # Apple Silicon GPU
        device = "mps"
        import platform
        device_name = f"Apple {platform.processor()}"
        
        return DeviceConfig(
            device=device,
            device_name=device_name,
            is_cuda=False,
            is_mps=True,
            is_cpu=False,
            dtype=torch.float32,  # MPS works best with float32
            batch_size=32,
            num_workers=4,
            memory_gb=0  # Shared memory, hard to detect
        )
    
    else:
        # CPU fallback
        import psutil
        memory_gb = psutil.virtual_memory().total / (1024**3)
        cpu_count = os.cpu_count() or 4
        
        return DeviceConfig(
            device="cpu",
            device_name=f"CPU ({cpu_count} cores)",
            is_cuda=False,
            is_mps=False,
            is_cpu=True,
            dtype=torch.float32,
            batch_size=16,
            num_workers=min(4, cpu_count),
            memory_gb=memory_gb
        )

# Detect and display device info
device_config = detect_device()

print("=" * 50)
print("DEVICE CONFIGURATION")
print("=" * 50)
print(f"Device: {device_config.device.upper()}")
print(f"Device Name: {device_config.device_name}")
print(f"Data Type: {device_config.dtype}")
print(f"Batch Size: {device_config.batch_size}")
print(f"Num Workers: {device_config.num_workers}")
if device_config.memory_gb > 0:
    print(f"Memory: {device_config.memory_gb:.1f} GB")
print("=" * 50)

## 3. Optimized Imports and Setup

In [ ]:
import lancedb
from datasets import load_dataset

# LlamaIndex core
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# Embeddings and vector store
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.lancedb import LanceDBVectorStore

# LLM integrations
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.llms.ollama import Ollama

# Async support
import nest_asyncio
nest_asyncio.apply()

print("All imports successful")

## 4. Optimized Embedding Model with Device Support

In [ ]:
class OptimizedEmbeddingModel:
    """
    Optimized embedding model wrapper that handles device placement,
    batching, and memory management.
    """
    
    # Model configurations with memory requirements
    MODELS = {
        "small": {
            "name": "BAAI/bge-small-en-v1.5",
            "dim": 384,
            "memory_mb": 130
        },
        "medium": {
            "name": "BAAI/bge-base-en-v1.5",
            "dim": 768,
            "memory_mb": 440
        },
        "large": {
            "name": "BAAI/bge-large-en-v1.5",
            "dim": 1024,
            "memory_mb": 1340
        }
    }
    
    def __init__(
        self,
        model_size: Literal["small", "medium", "large"] = "small",
        device_config: Optional[DeviceConfig] = None,
        trust_remote_code: bool = True
    ):
        self.device_config = device_config or detect_device()
        self.model_info = self.MODELS[model_size]
        
        print(f"Initializing {model_size} embedding model on {self.device_config.device}...")
        
        # Configure model kwargs based on device
        model_kwargs = {"device": self.device_config.device}
        
        # Add precision settings for CUDA
        if self.device_config.is_cuda:
            model_kwargs["torch_dtype"] = self.device_config.dtype
        
        # Encode kwargs for optimization
        encode_kwargs = {
            "normalize_embeddings": True,
            "batch_size": self.device_config.batch_size
        }
        
        self.embed_model = HuggingFaceEmbedding(
            model_name=self.model_info["name"],
            model_kwargs=model_kwargs,
            encode_kwargs=encode_kwargs,
            trust_remote_code=trust_remote_code
        )
        
        print(f"Model loaded: {self.model_info['name']}")
        print(f"Embedding dimension: {self.model_info['dim']}")
    
    def get_model(self) -> HuggingFaceEmbedding:
        """Return the underlying embedding model"""
        return self.embed_model
    
    def get_embedding(self, text: str) -> List[float]:
        """Get embedding for a single text"""
        return self.embed_model.get_text_embedding(text)
    
    def get_embeddings_batch(self, texts: List[str]) -> List[List[float]]:
        """Get embeddings for a batch of texts with memory management"""
        embeddings = []
        batch_size = self.device_config.batch_size
        
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_embeddings = self.embed_model.get_text_embedding_batch(batch)
            embeddings.extend(batch_embeddings)
            
            # Clear GPU cache periodically
            if self.device_config.is_cuda and (i + batch_size) % (batch_size * 10) == 0:
                torch.cuda.empty_cache()
        
        return embeddings
    
    def clear_cache(self):
        """Clear GPU/CPU cache"""
        gc.collect()
        if self.device_config.is_cuda:
            torch.cuda.empty_cache()


def create_optimized_embed_model(
    model_size: str = "small",
    device_config: Optional[DeviceConfig] = None
) -> HuggingFaceEmbedding:
    """
    Factory function to create an optimized embedding model.
    """
    optimizer = OptimizedEmbeddingModel(
        model_size=model_size,
        device_config=device_config or detect_device()
    )
    return optimizer.get_model()


# Create optimized embedding model
embed_model = create_optimized_embed_model("small", device_config)

## 5. Performance Monitoring Utilities

In [ ]:
class PerformanceMonitor:
    """
    Monitor and track performance metrics for CPU/GPU operations.
    """
    
    def __init__(self, device_config: DeviceConfig):
        self.device_config = device_config
        self.metrics: Dict[str, List[Dict]] = {}
    
    def benchmark(self, name: str):
        """Decorator to benchmark a function"""
        def decorator(func):
            @wraps(func)
            def wrapper(*args, **kwargs):
                # Sync GPU if available
                if self.device_config.is_cuda:
                    torch.cuda.synchronize()
                
                start_time = time.perf_counter()
                start_memory = self._get_memory_usage()
                
                result = func(*args, **kwargs)
                
                if self.device_config.is_cuda:
                    torch.cuda.synchronize()
                
                end_time = time.perf_counter()
                end_memory = self._get_memory_usage()
                
                metric = {
                    "duration_ms": (end_time - start_time) * 1000,
                    "memory_delta_mb": end_memory - start_memory,
                    "peak_memory_mb": end_memory
                }
                
                if name not in self.metrics:
                    self.metrics[name] = []
                self.metrics[name].append(metric)
                
                return result
            return wrapper
        return decorator
    
    def _get_memory_usage(self) -> float:
        """Get current memory usage in MB"""
        if self.device_config.is_cuda:
            return torch.cuda.memory_allocated() / (1024**2)
        else:
            import psutil
            return psutil.Process().memory_info().rss / (1024**2)
    
    def print_summary(self):
        """Print performance summary"""
        print("\n" + "=" * 60)
        print("PERFORMANCE SUMMARY")
        print("=" * 60)
        print(f"Device: {self.device_config.device_name}")
        print("-" * 60)
        
        for name, measurements in self.metrics.items():
            if measurements:
                durations = [m["duration_ms"] for m in measurements]
                avg_duration = np.mean(durations)
                std_duration = np.std(durations) if len(durations) > 1 else 0
                
                print(f"\n{name}:")
                print(f"  Avg Duration: {avg_duration:.2f} ms (+/- {std_duration:.2f})")
                print(f"  Total Calls: {len(measurements)}")
                print(f"  Total Time: {sum(durations):.2f} ms")
        
        print("\n" + "=" * 60)

# Initialize performance monitor
perf_monitor = PerformanceMonitor(device_config)
print("Performance monitor initialized")

## 6. Data Loading with Optimizations

In [ ]:
def prepare_documents(
    num_samples: int = 100,
    dataset_name: str = "dvilasuero/finepersonas-v0.1-tiny"
) -> List[Document]:
    """
    Load and prepare documents from HuggingFace dataset.
    """
    print(f"Loading {num_samples} samples from {dataset_name}...")
    
    dataset = load_dataset(dataset_name, split="train")
    
    documents = []
    for i, item in enumerate(dataset.select(range(min(num_samples, len(dataset))))):
        doc = Document(
            text=item["persona"],
            metadata={
                "persona_id": i,
                "source": "finepersonas-dataset"
            }
        )
        documents.append(doc)
    
    print(f"Prepared {len(documents)} documents")
    return documents

# Load documents
documents = prepare_documents(num_samples=100)

## 7. Optimized LanceDB Setup

In [ ]:
class OptimizedVectorStore:
    """
    Optimized vector store wrapper with device-aware operations.
    """
    
    def __init__(
        self,
        uri: str = "./lancedb_optimized",
        table_name: str = "optimized_rag",
        device_config: Optional[DeviceConfig] = None
    ):
        self.uri = uri
        self.table_name = table_name
        self.device_config = device_config or detect_device()
        
        # Connect to LanceDB
        self.db = lancedb.connect(uri)
        
        print(f"Connected to LanceDB at {uri}")
    
    def create_vector_store(self, mode: str = "overwrite") -> LanceDBVectorStore:
        """
        Create a LanceDB vector store with optimized settings.
        """
        vector_store = LanceDBVectorStore(
            uri=self.uri,
            table_name=self.table_name,
            mode=mode,
            nprobes=20  # Balance between speed and accuracy
        )
        return vector_store
    
    def get_table_stats(self):
        """Get statistics about the vector store table"""
        try:
            table = self.db.open_table(self.table_name)
            return {
                "total_rows": table.count_rows(),
                "schema": str(table.schema)
            }
        except Exception as e:
            return {"error": str(e)}

# Initialize optimized vector store
optimized_store = OptimizedVectorStore(
    uri="./lancedb_optimized",
    table_name="personas_optimized",
    device_config=device_config
)

vector_store = optimized_store.create_vector_store()

## 8. Optimized Ingestion Pipeline

In [ ]:
async def optimized_ingestion(
    documents: List[Document],
    vector_store: LanceDBVectorStore,
    embed_model: HuggingFaceEmbedding,
    device_config: DeviceConfig,
    chunk_size: int = 512,
    chunk_overlap: int = 50
) -> int:
    """
    Optimized document ingestion with device-aware batching.
    """
    print(f"\nStarting optimized ingestion on {device_config.device}...")
    print(f"Chunk size: {chunk_size}, Overlap: {chunk_overlap}")
    print(f"Batch size: {device_config.batch_size}")
    
    start_time = time.perf_counter()
    
    # Create pipeline with optimized settings
    pipeline = IngestionPipeline(
        transformations=[
            SentenceSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap
            ),
            embed_model,
        ],
        vector_store=vector_store,
    )
    
    # Process in batches for memory efficiency
    total_nodes = 0
    batch_size = device_config.batch_size * 2  # Document batch size
    
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i + batch_size]
        nodes = await pipeline.arun(documents=batch)
        total_nodes += len(nodes)
        
        # Progress update
        progress = min((i + batch_size) / len(documents) * 100, 100)
        print(f"Progress: {progress:.1f}% ({total_nodes} nodes processed)", end="\r")
        
        # Memory cleanup for GPU
        if device_config.is_cuda:
            torch.cuda.empty_cache()
    
    elapsed = time.perf_counter() - start_time
    
    print(f"\n\nIngestion complete!")
    print(f"Total nodes: {total_nodes}")
    print(f"Time elapsed: {elapsed:.2f} seconds")
    print(f"Throughput: {total_nodes/elapsed:.1f} nodes/second")
    
    return total_nodes

# Run optimized ingestion
num_nodes = await optimized_ingestion(
    documents=documents,
    vector_store=vector_store,
    embed_model=embed_model,
    device_config=device_config
)

## 9. Optimized Query Engine

In [ ]:
class OptimizedQueryEngine:
    """
    Optimized query engine with device-aware processing.
    """
    
    def __init__(
        self,
        vector_store: LanceDBVectorStore,
        embed_model: HuggingFaceEmbedding,
        device_config: DeviceConfig,
        llm: Optional[Any] = None
    ):
        self.vector_store = vector_store
        self.embed_model = embed_model
        self.device_config = device_config
        self.llm = llm
        
        # Create index
        self.index = VectorStoreIndex.from_vector_store(
            vector_store=vector_store,
            embed_model=embed_model
        )
        
        # Create query engine
        engine_kwargs = {"response_mode": "tree_summarize"}
        if llm:
            engine_kwargs["llm"] = llm
        
        self.query_engine = self.index.as_query_engine(**engine_kwargs)
    
    def vector_search(
        self,
        query: str,
        top_k: int = 5
    ) -> List[Dict]:
        """
        Perform optimized vector search without LLM.
        """
        # Get query embedding
        query_embedding = self.embed_model.get_text_embedding(query)
        
        # Search in LanceDB
        db = lancedb.connect(self.vector_store.uri)
        table = db.open_table(self.vector_store.table_name)
        
        results = table.search(query_embedding).limit(top_k).to_pandas()
        
        return results.to_dict('records')
    
    def query(self, question: str):
        """
        Query with full RAG pipeline.
        """
        return self.query_engine.query(question)


# Create optimized query engine (without LLM for now)
query_engine = OptimizedQueryEngine(
    vector_store=vector_store,
    embed_model=embed_model,
    device_config=device_config
)

print("Query engine initialized")

## 10. Vector Search Testing

In [ ]:
def benchmark_vector_search(
    query_engine: OptimizedQueryEngine,
    queries: List[str],
    top_k: int = 5,
    num_iterations: int = 3
):
    """
    Benchmark vector search performance.
    """
    print("\n" + "=" * 60)
    print("VECTOR SEARCH BENCHMARK")
    print("=" * 60)
    print(f"Device: {query_engine.device_config.device_name}")
    print(f"Queries: {len(queries)}, Iterations: {num_iterations}")
    print("-" * 60)
    
    all_times = []
    
    for query in queries:
        query_times = []
        
        for _ in range(num_iterations):
            # Warm up GPU if needed
            if query_engine.device_config.is_cuda:
                torch.cuda.synchronize()
            
            start = time.perf_counter()
            results = query_engine.vector_search(query, top_k=top_k)
            
            if query_engine.device_config.is_cuda:
                torch.cuda.synchronize()
            
            elapsed = (time.perf_counter() - start) * 1000  # ms
            query_times.append(elapsed)
        
        avg_time = np.mean(query_times)
        all_times.append(avg_time)
        
        print(f"\nQuery: '{query[:40]}...'")
        print(f"  Avg Time: {avg_time:.2f} ms")
        print(f"  Results: {len(results)}")
        
        # Show top result
        if results:
            top_result = results[0]
            score = top_result.get('_distance', 'N/A')
            text = top_result.get('text', '')[:100]
            print(f"  Top Match (score={score:.3f}): {text}...")
    
    print("\n" + "-" * 60)
    print(f"Overall Average: {np.mean(all_times):.2f} ms per query")
    print("=" * 60)

# Test queries
test_queries = [
    "technology and artificial intelligence expert",
    "teacher educator professor",
    "environment climate sustainability",
    "art culture heritage creative"
]

benchmark_vector_search(query_engine, test_queries)

## 11. RAG with LLM (HuggingFace API)

In [ ]:
# Configure your HuggingFace API token
os.environ["HUGGINGFACE_API_KEY"] = "your_token_here"  # Replace with your token

async def test_rag_with_huggingface():
    """
    Test RAG with HuggingFace API.
    """
    print("\n" + "=" * 60)
    print("RAG WITH HUGGINGFACE API")
    print("=" * 60)
    
    try:
        # Initialize LLM
        llm = HuggingFaceInferenceAPI(
            model_name="HuggingFaceTB/SmolLM3-3B",
            token=os.environ.get("HUGGINGFACE_API_KEY")
        )
        
        # Create query engine with LLM
        rag_engine = OptimizedQueryEngine(
            vector_store=vector_store,
            embed_model=embed_model,
            device_config=device_config,
            llm=llm
        )
        
        # Test query
        query = "Find personas interested in technology and AI"
        print(f"\nQuery: {query}")
        print("-" * 40)
        
        response = rag_engine.query(query)
        print(f"Response: {response}")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Make sure to set your HuggingFace API token")

# Uncomment to test
# await test_rag_with_huggingface()

## 12. RAG with Local LLM (Ollama)

In [ ]:
async def test_rag_with_ollama(model_name: str = "llama3.2:1b"):
    """
    Test RAG with local Ollama LLM.
    """
    print("\n" + "=" * 60)
    print("RAG WITH LOCAL LLM (OLLAMA)")
    print("=" * 60)
    print(f"Model: {model_name}")
    print(f"Embedding Device: {device_config.device}")
    
    try:
        # Initialize Ollama LLM
        llm = Ollama(
            model=model_name,
            base_url="http://localhost:11434",
            request_timeout=120.0
        )
        
        # Create query engine with LLM
        rag_engine = OptimizedQueryEngine(
            vector_store=vector_store,
            embed_model=embed_model,
            device_config=device_config,
            llm=llm
        )
        
        # Test queries
        queries = [
            "Find personas interested in technology and AI",
            "Who are the educators in the dataset?",
            "Describe environmental professionals"
        ]
        
        for query in queries:
            print(f"\nQuery: {query}")
            print("-" * 40)
            
            start = time.perf_counter()
            response = rag_engine.query(query)
            elapsed = time.perf_counter() - start
            
            print(f"Response ({elapsed:.2f}s): {response}")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Make sure Ollama is running: ollama serve")

# Uncomment to test (requires Ollama running)
# await test_rag_with_ollama()

## 13. Memory Optimization Utilities

In [ ]:
def memory_status(device_config: DeviceConfig):
    """
    Display current memory status.
    """
    print("\n" + "=" * 40)
    print("MEMORY STATUS")
    print("=" * 40)
    
    if device_config.is_cuda:
        allocated = torch.cuda.memory_allocated() / (1024**3)
        reserved = torch.cuda.memory_reserved() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        
        print(f"GPU Memory:")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved: {reserved:.2f} GB")
        print(f"  Total: {total:.2f} GB")
        print(f"  Free: {total - reserved:.2f} GB")
    
    else:
        import psutil
        mem = psutil.virtual_memory()
        
        print(f"System Memory:")
        print(f"  Used: {mem.used / (1024**3):.2f} GB")
        print(f"  Available: {mem.available / (1024**3):.2f} GB")
        print(f"  Total: {mem.total / (1024**3):.2f} GB")
        print(f"  Percent: {mem.percent}%")
    
    print("=" * 40)

def clear_memory(device_config: DeviceConfig):
    """
    Aggressively clear memory.
    """
    gc.collect()
    
    if device_config.is_cuda:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    print("Memory cleared")

# Show current memory status
memory_status(device_config)

## 14. Batch Processing for Large Datasets

In [ ]:
async def process_large_dataset(
    documents: List[Document],
    device_config: DeviceConfig,
    batch_size: Optional[int] = None,
    table_prefix: str = "large_dataset"
):
    """
    Process large datasets with memory-efficient batching.
    """
    batch_size = batch_size or (device_config.batch_size * 4)
    total_batches = (len(documents) + batch_size - 1) // batch_size
    
    print(f"\nProcessing {len(documents)} documents in {total_batches} batches")
    print(f"Batch size: {batch_size}")
    print(f"Device: {device_config.device}")
    
    # Create embedding model for batching
    batch_embed_model = create_optimized_embed_model("small", device_config)
    
    total_nodes = 0
    start_time = time.perf_counter()
    
    for batch_idx in range(total_batches):
        batch_start = batch_idx * batch_size
        batch_end = min(batch_start + batch_size, len(documents))
        batch_docs = documents[batch_start:batch_end]
        
        # Create table for this batch
        table_name = f"{table_prefix}_batch_{batch_idx}"
        
        batch_store = LanceDBVectorStore(
            uri="./lancedb_optimized",
            table_name=table_name,
            mode="overwrite"
        )
        
        pipeline = IngestionPipeline(
            transformations=[
                SentenceSplitter(chunk_size=512, chunk_overlap=50),
                batch_embed_model,
            ],
            vector_store=batch_store,
        )
        
        nodes = await pipeline.arun(documents=batch_docs)
        total_nodes += len(nodes)
        
        # Memory cleanup
        clear_memory(device_config)
        
        progress = (batch_idx + 1) / total_batches * 100
        print(f"Batch {batch_idx + 1}/{total_batches} complete ({progress:.1f}%)")
    
    elapsed = time.perf_counter() - start_time
    
    print(f"\nLarge dataset processing complete!")
    print(f"Total nodes: {total_nodes}")
    print(f"Total time: {elapsed:.2f} seconds")
    print(f"Throughput: {total_nodes/elapsed:.1f} nodes/second")
    
    return total_nodes

# Example usage (uncomment to test with more data)
# large_documents = prepare_documents(num_samples=500)
# await process_large_dataset(large_documents, device_config)

## 15. Compare CPU vs GPU Performance

In [ ]:
def compare_devices():
    """
    Compare performance between CPU and GPU (if available).
    """
    print("\n" + "=" * 60)
    print("CPU VS GPU COMPARISON")
    print("=" * 60)
    
    test_texts = [
        "This is a test sentence for embedding generation.",
        "Another example text to process and embed.",
        "Technology and artificial intelligence are transforming industries."
    ] * 10  # 30 texts
    
    results = {}
    
    # Test CPU
    print("\nTesting CPU...")
    cpu_config = DeviceConfig(
        device="cpu",
        device_name="CPU",
        is_cuda=False,
        is_mps=False,
        is_cpu=True,
        dtype=torch.float32,
        batch_size=16,
        num_workers=4,
        memory_gb=0
    )
    
    cpu_model = create_optimized_embed_model("small", cpu_config)
    
    start = time.perf_counter()
    for text in test_texts:
        _ = cpu_model.get_text_embedding(text)
    cpu_time = time.perf_counter() - start
    
    results["CPU"] = cpu_time
    print(f"  Time: {cpu_time:.3f}s for {len(test_texts)} embeddings")
    print(f"  Rate: {len(test_texts)/cpu_time:.1f} embeddings/second")
    
    # Test GPU if available
    if torch.cuda.is_available():
        print("\nTesting GPU...")
        gpu_config = detect_device()  # Will return GPU config
        gpu_model = create_optimized_embed_model("small", gpu_config)
        
        # Warm up
        _ = gpu_model.get_text_embedding(test_texts[0])
        torch.cuda.synchronize()
        
        start = time.perf_counter()
        for text in test_texts:
            _ = gpu_model.get_text_embedding(text)
        torch.cuda.synchronize()
        gpu_time = time.perf_counter() - start
        
        results["GPU"] = gpu_time
        print(f"  Time: {gpu_time:.3f}s for {len(test_texts)} embeddings")
        print(f"  Rate: {len(test_texts)/gpu_time:.1f} embeddings/second")
        
        speedup = cpu_time / gpu_time
        print(f"\nGPU Speedup: {speedup:.2f}x faster than CPU")
    
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("\nTesting MPS (Apple Silicon)...")
        mps_config = detect_device()
        mps_model = create_optimized_embed_model("small", mps_config)
        
        start = time.perf_counter()
        for text in test_texts:
            _ = mps_model.get_text_embedding(text)
        mps_time = time.perf_counter() - start
        
        results["MPS"] = mps_time
        print(f"  Time: {mps_time:.3f}s for {len(test_texts)} embeddings")
        print(f"  Rate: {len(test_texts)/mps_time:.1f} embeddings/second")
        
        speedup = cpu_time / mps_time
        print(f"\nMPS Speedup: {speedup:.2f}x faster than CPU")
    
    else:
        print("\nNo GPU available for comparison")
    
    print("=" * 60)
    return results

# Run comparison
comparison_results = compare_devices()

## 16. Quick Start Functions

In [ ]:
async def quick_start_rag(
    num_samples: int = 100,
    model_size: str = "small",
    force_cpu: bool = False
):
    """
    Quick start function for RAG setup.
    
    Args:
        num_samples: Number of documents to load
        model_size: Embedding model size (small, medium, large)
        force_cpu: Force CPU usage even if GPU is available
    """
    print("\n" + "=" * 60)
    print("QUICK START RAG")
    print("=" * 60)
    
    # Device setup
    if force_cpu:
        config = DeviceConfig(
            device="cpu",
            device_name="CPU (forced)",
            is_cuda=False,
            is_mps=False,
            is_cpu=True,
            dtype=torch.float32,
            batch_size=16,
            num_workers=4,
            memory_gb=0
        )
    else:
        config = detect_device()
    
    print(f"Device: {config.device_name}")
    
    # Load documents
    docs = prepare_documents(num_samples)
    
    # Create embedding model
    embed = create_optimized_embed_model(model_size, config)
    
    # Create vector store
    store = OptimizedVectorStore(
        uri="./lancedb_quickstart",
        table_name="quickstart",
        device_config=config
    )
    vs = store.create_vector_store()
    
    # Ingest documents
    await optimized_ingestion(docs, vs, embed, config)
    
    # Create query engine
    engine = OptimizedQueryEngine(vs, embed, config)
    
    print("\nRAG system ready!")
    print("Use: engine.vector_search('your query')")
    print("=" * 60)
    
    return engine, config

# Uncomment to run quick start
# engine, config = await quick_start_rag(num_samples=100)

## Summary

This optimized notebook provides:

### Automatic Device Detection
- CUDA (NVIDIA GPUs)
- MPS (Apple Silicon)
- CPU fallback

### Performance Optimizations
- Mixed precision (FP16/BF16) for GPU
- Adaptive batch sizing
- Memory-efficient processing
- Async operations

### Features
- Performance benchmarking
- Memory monitoring
- Large dataset processing
- CPU vs GPU comparison

### Usage
```python
# Quick start
engine, config = await quick_start_rag(num_samples=100)

# Vector search
results = engine.vector_search("your query")

# Full RAG with LLM
response = engine.query("your question")
```